In [26]:
import pandas as pd
df_index=pd.read_csv("/media/zihengc/T7 Shield/mpra/MPRA_design/Get_sequences/Barcoding/2022.2.7 AD_MPRA_FINAL_INDELCORRECT/AD_MPRA_2_7/index20230213_curated.csv")
df_index
index_ref = df_index[df_index["allele"]=="ref"]
index_alt = df_index[df_index["allele"]=="alt"]
len(index_alt)+len(index_ref)

abrt = []
one_ref_dic = {}
abrt2=[]
for i in range(1,len(index_alt)):
    alt_tmp = index_alt.iloc[i]
    rsid = alt_tmp["rsid"]
    chr = alt_tmp["chr"]
    start = alt_tmp["start"]
    end = alt_tmp["end"]
    center = alt_tmp["center"]
    summit = alt_tmp["summit"]
    snp = alt_tmp["snp"]
    a1 = alt_tmp["a1"]
    
    childrens_df = index_ref[(index_ref["start"]==start)&(index_ref["end"]==end)&(index_ref["chr"]==chr)& (index_ref["center"]==center)&\
        (index_ref["rsid"]==rsid)&(index_ref["a1"]==a1)&(index_ref["summit"]==summit)]

    #if alt_tmp["full"]=='alt:rs77795057:PEAKCENTER:chr6:119110971:A:G:119110832:119111055:119110898':
        #break
    
    if len(childrens_df)!=1:
        #abrt.append(alt_tmp)

        if len(childrens_df)==0:
            childrens_df2 = index_ref[(index_ref["start"]==start)&(index_ref["end"]==end)&(index_ref["chr"]==chr)& (index_ref["center"]=="PEAKCENTER")&(index_ref["summit"]==summit)]
            if len(childrens_df2) ==1:
                one_ref_dic[alt_tmp["full"]] = childrens_df2["full"].tolist()[0]

            elif len(childrens_df2) ==2:
                for row in childrens_df2.iterrows():
                    if len(row[1]["a2"]) == len(alt_tmp["a2"]):
                        one_ref_dic[alt_tmp["full"]] = row[1]["full"]
            else:
                #print(alt_tmp["full"])
                abrt2.append(alt_tmp)
    else:
        one_ref_dic[alt_tmp["full"]] = childrens_df["full"].tolist()[0]

#special case
one_ref_dic["alt:rs1198397268:SNPCENTER:chr19:1999196:CAG:C:1999083:1999309:1999196"] = 'ref:rs965034941:SNPCENTER:chr19:1999195:CCA:C:1999082:1999308:1999195'


print(len(one_ref_dic),len(index_alt),len(abrt2))


abrt3=[]
abrt3_ref = []
for alt_tmp in abrt2:
    
    rsid = alt_tmp["rsid"]
    if rsid =="rs1198397268":
        continue
    chr = alt_tmp["chr"]
    start = alt_tmp["start"]
    end = alt_tmp["end"]
    center = alt_tmp["center"]
    summit = alt_tmp["summit"]
    snp = alt_tmp["snp"]
    a1 = alt_tmp["a1"]
    if summit[-12:] == "motifdisrupt":
        center_new= "PEAKCENTER"
        summit_new=summit[:-13]
    elif summit[-10:] == "PEAKCENTER":
        center_new = "PEAKCENTER"
        summit_new = summit[:-24]
    elif summit[-9:] == "SNPCENTER":
        center_new = "SNPCENTER"
        summit_new = summit[:-23]
    else:
        print("error", summit)

    childrens_df = index_ref[(index_ref["start"]==start)&(index_ref["end"]==end)&(index_ref["chr"]==chr)&(index_ref["summit"]==summit_new)&(index_ref["rsid"]==rsid)]
    
    if len(childrens_df)!=1:
        childrens_df = index_ref[(index_ref["start"]==start)&(index_ref["end"]==end)&(index_ref["chr"]==chr)&(index_ref["center"]==center_new)&(index_ref["summit"]==summit_new)]

        if len(childrens_df)==1:
            one_ref_dic[alt_tmp["full"]] = childrens_df["full"].tolist()[0]
        elif len(childrens_df)==2:
            childrens_df=childrens_df[childrens_df["rsid"]==rsid]
            childrens_df=childrens_df[childrens_df["a2"].str.len()==1]
            if len(childrens_df) == 1:
                one_ref_dic[alt_tmp["full"]] = childrens_df["full"].tolist()[0]
            else:
                abrt3.append(alt_tmp)
                abrt3_ref.append(childrens_df)
        else:
            #print(len(childrens_df2))
            abrt3.append(alt_tmp)
            abrt3_ref.append(childrens_df)

    else:
        one_ref_dic[alt_tmp["full"]] = childrens_df["full"].tolist()[0]

    alt_tmp=None

for i in index_alt["full"]:
    if i not in one_ref_dic.keys():
        print(i)


import csv
with open("ALT_REF_LookUpTable_20231117.csv","w") as f:
    for key in one_ref_dic.keys():
        f.write("%s,%s\n"%(key,one_ref_dic[key]))

868 1033 165
PanTissueControl.SV40_224bp_1xEnh_Promoter


In [25]:
import pandas as pd
df = pd.read_csv('/media/zihengc/T7 Shield/mpra3_lib_analysis/indexing/ALT_REF_LookUpTable_filtered_20231111.csv')
df.columns = ['ALT','REF']
#filter out exact match between alt and ref
df2 = df[df["ALT"].str.slice(4,)!=df["REF"].str.slice(4,)]
coordinate_equal = df2["ALT"].str.split(":").apply(lambda x:x[-3:])==df2["REF"].str.split(":").apply(lambda x:x[-3:])
center_equal = df2["ALT"].str.split(":").apply(lambda x:x[2])==df2["REF"].str.split(":").apply(lambda x:x[2])

df2[~(coordinate_equal&center_equal)]#.to_csv("ALT_REF_LookUpTable_MandatoryAmendment_20231117_unannotated.csv")

In [4]:
#swap several pairs
import pandas as pd
df_amendment = pd.read_csv('/media/zihengc/T7 Shield/mpra3_lib_analysis/indexing/archived/ALT_REF_LookUpTable_MandatoryAmendment_20231117.csv',index_col=0)
df_amendment = df_amendment[df_amendment["Mismatch_Visual_Exam"] == "F"]
df_tochange=pd.read_csv('/media/zihengc/T7 Shield/mpra3_lib_analysis/indexing/archived/ALT_REF_LookUpTable_20231111.csv' ,index_col=0)
df_tochange.loc[df_amendment.index, '1'] = df_amendment['REF'].tolist()
df_tochange.to_csv('/media/zihengc/T7 Shield/mpra3_lib_analysis/indexing/ALT_REF_LookUpTable_amended_20231117.csv')

In [5]:
#swap several pairs
import pandas as pd
df_amendment = pd.read_csv('/media/zihengc/T7 Shield/mpra3_lib_analysis/indexing/archived/ALT_REF_LookUpTable_MandatoryAmendment_20231117.csv',index_col=0)
df_amendment = df_amendment[df_amendment["Mismatch_Visual_Exam"] == "F"]
df_tochange=pd.read_csv('/media/zihengc/T7 Shield/mpra3_lib_analysis/indexing/archived/ALT_REF_LookUpTable_filtered_20231111.csv' ,index_col=0)
df_tochange.loc[df_amendment.index, '1'] = df_amendment['REF'].tolist()
df_tochange.to_csv('/media/zihengc/T7 Shield/mpra3_lib_analysis/indexing/ALT_REF_LookUpTable_filtered_amended_20231117.csv')



In [1]:
#separate ALLELE and MOTIFDISRUPT
'''
import pandas as pd
df=pd.read_csv("indexing/ALT_REF_LookUpTable_filtered_amended_20231117.csv",index_col=0)
df_allele = df[df.index.str.contains("alt")]
df_motif = df[df.index.str.contains("motifdisrupt")]
df[df.index.str.contains("alt")].to_csv("indexing/ALT_REF_LookUpTable_filtered_amended_alleleOnly_20240605.csv")
df[df.index.str.contains("motifdisrupt")].to_csv("indexing/ALT_REF_LookUpTable_filtered_amended_motifOnly_20240605.csv")
df_motif_allele = pd.merge(df_motif,df_allele,left_on="1",right_on="1")
df_motif_allele.columns = ["motif","ref","alt"]
'''


In [60]:
from collections import defaultdict
df_motif_allele=pd.read_csv("indexing/ALT_REF_LookUpTable_filtered_amended_motif_alt_ref_20240127.csv")
# Initialize a defaultdict of sets
grouped_dict = defaultdict(set)

# Iterate over each row in the dataframe
for _, row in df_motif_allele.iterrows():
    grouped_dict[row['motif']].add(row['ref'])
    grouped_dict[row['motif']].add(row['alt'])

# Convert defaultdict to a regular dictionary
grouped_dict = {k: list(v) for k, v in grouped_dict.items()}

print("maximum number of elements that have the same ref enhancer", max(len(v) for v in grouped_dict.values()))

maximum number of elements that have the same ref enhancer 4
